In [ ]:
from argparse import Namespace
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.build import build_dataset
from src.misc import load_config
from src.predict import AnchorSlice, load_predict_config, load_predictor


def pick_anchor(ds, axis):
    idx = ds.condition_indices[axis]
    pos = int(torch.randint(len(idx), ()).item())
    return ds[idx[pos]][0][0].numpy().astype("uint8")

In [ ]:
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"
ANCHOR_AXIS = 0
AXES = ("axis 0", "axis 1", "axis 2")
OFFSETS = (-2, -1, 0, 1, 2)

In [ ]:
cfg = load_predict_config(PREDICT_CONFIG)
run = ROOT / cfg.run_dir
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pred = load_predictor(run, device=dev)
opts = cfg.make_options(pred)

data = Namespace(**load_config(run / "model.yaml"))
data.data_dir = {
    axis: (path if (path := Path(value)).is_absolute() else run / path).resolve()
    for axis, value in data.data_dir.items()
}
data.augment = False
ds = build_dataset(data)
img = pick_anchor(ds, ANCHOR_AXIS)
anc = AnchorSlice(
    img,
    axis=ANCHOR_AXIS,
    index=opts.volume_size // 2,
)

In [ ]:
start = time.perf_counter()
vol, stats = pred.predict(opts, anchors=[anc])
secs = time.perf_counter() - start
fracs = [round(v, 4) for v in stats["phase_fractions"]]
print(
    f"volume={tuple(vol.shape)} anchor={anc.index} "
    f"elapsed={secs:.1f}s fractions={fracs}"
)

In [ ]:
center = opts.volume_size // 2
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for dim, ax in enumerate(axes):
    ax.imshow(
        vol.select(dim, center),
        cmap="gray",
        vmin=0,
        vmax=opts.num_phases - 1,
        interpolation="nearest",
    )
    ax.set_title(f"{AXES[dim]} / CENTER", fontsize=16, fontweight="bold")
    ax.axis("off")
fig.suptitle("Orthogonal center slices", fontsize=20, fontweight="bold")
fig.subplots_adjust(top=0.82, wspace=0.08)
plt.show()

In [ ]:
near = [
    vol.select(anc.axis, anc.index + offset).numpy()
    for offset in OFFSETS
]
panels = [(anc.image, "ANCHOR INPUT", True)] + [
    (
        image,
        "GENERATED AT ANCHOR" if offset == 0 else f"NEIGHBOR {offset:+d}",
        offset == 0,
    )
    for offset, image in zip(OFFSETS, near)
]

fig, axes = plt.subplots(2, 3, figsize=(11, 8))
for ax, (img, title, marked) in zip(axes.ravel(), panels):
    ax.imshow(
        img,
        cmap="gray",
        vmin=0,
        vmax=opts.num_phases - 1,
        interpolation="nearest",
    )
    ax.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        color="crimson" if marked else "black",
    )
    ax.axis("off")
fig.suptitle(
    f"Anchor neighborhood / {AXES[anc.axis]} / index {anc.index}",
    fontsize=20,
    fontweight="bold",
)
fig.subplots_adjust(top=0.88, hspace=0.25, wspace=0.08)
plt.show()

In [ ]:
%gui qt
import napari

viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(vol.numpy(), name="MPDD volume")
viewer